# Initial-State Landscape Slice

This notebook draws a 2D landscape slice by sampling different initial latent states.

Important caveat:
- this is not the full 5280-D landscape
- it is a 2D plane through the initial latent state space
- height is defined from the recurrent residual `f_t - f_{t-1}`
- by default, we visualize the first generated token after a fixed prompt


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_DIR = Path("/data/hansen/serv12/HRM-Deq/models/huginn-0125")
DEVICE = "cuda:0"
DTYPE = torch.bfloat16

# Keep these modest at first; larger grids get expensive.
NUM_STEPS = 16
GRID_SIZE = 15
RADIUS_MULTIPLIER = 1.0
SEED = 7

QUESTION = (
    "There are 15 trees in the grove. Grove workers will plant trees in the grove today. "
    "After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
)
PROMPT_TEXT = f"Q: {QUESTION}\n\nA:"

torch.manual_seed(SEED)
assert MODEL_DIR.exists(), MODEL_DIR


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    device_map={"": DEVICE},
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model.eval()

prompt_ids = tokenizer(PROMPT_TEXT, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
embedded_inputs, block_idx = model.embed_inputs(prompt_ids)
base_state = model.initialize_state(embedded_inputs)
base_last = base_state[:, -1:, :].detach().clone()

print("prompt length:", prompt_ids.shape[1])
print("embedded_inputs:", tuple(embedded_inputs.shape))
print("base_state:", tuple(base_state.shape))
print("base_last norm:", torch.linalg.vector_norm(base_last.float(), dim=-1).item())


In [ ]:
def make_orthonormal_directions(reference: torch.Tensor):
    u = torch.randn_like(reference)
    u = u / torch.linalg.vector_norm(u.float(), dim=-1, keepdim=True).clamp_min(1e-6).to(u.dtype)

    v = torch.randn_like(reference)
    proj = (u.float() * v.float()).sum(dim=-1, keepdim=True)
    v = v - proj.to(v.dtype) * u
    v = v / torch.linalg.vector_norm(v.float(), dim=-1, keepdim=True).clamp_min(1e-6).to(v.dtype)
    return u, v


u, v = make_orthonormal_directions(base_last)
base_radius = torch.linalg.vector_norm(base_last.float(), dim=-1).mean().item() * RADIUS_MULTIPLIER
axis = torch.linspace(-base_radius, base_radius, GRID_SIZE, device=DEVICE, dtype=torch.float32)
xx, yy = torch.meshgrid(axis, axis, indexing="xy")

alphas = xx.reshape(-1, 1, 1, 1)
betas = yy.reshape(-1, 1, 1, 1)
num_points = alphas.shape[0]

candidate_last = base_last.unsqueeze(0) + alphas * u.unsqueeze(0) + betas * v.unsqueeze(0)
candidate_last = candidate_last.squeeze(1).to(DTYPE)

batched_embedded_inputs = embedded_inputs.repeat(num_points, 1, 1)
batched_state = base_state.repeat(num_points, 1, 1)
batched_state[:, -1:, :] = candidate_last

print("grid points:", num_points)
print("candidate_last:", tuple(candidate_last.shape))


In [ ]:
@torch.inference_mode()
def rollout_landscape(model, embedded_inputs, initial_state, block_idx, num_steps):
    current = initial_state.clone()
    local_block_idx = block_idx.clone()
    step_deltas = []

    for step in range(num_steps):
        prev_last = current[:, -1:, :].clone()
        current, local_block_idx, _ = model.iterate_one_step(
            embedded_inputs,
            current,
            block_idx=local_block_idx,
            current_step=step,
        )
        delta = current[:, -1:, :] - prev_last
        step_deltas.append(torch.linalg.vector_norm(delta.float(), dim=-1).squeeze(-1))

    step_deltas = torch.stack(step_deltas, dim=1)  # [batch, step]
    return {
        "step_deltas": step_deltas,
        "final_step_delta": step_deltas[:, -1],
        "avg_step_delta": step_deltas.mean(dim=1),
        "path_length": step_deltas.sum(dim=1),
    }


landscape = rollout_landscape(
    model=model,
    embedded_inputs=batched_embedded_inputs,
    initial_state=batched_state,
    block_idx=block_idx,
    num_steps=NUM_STEPS,
)

final_step_delta = landscape["final_step_delta"].reshape(GRID_SIZE, GRID_SIZE).cpu().numpy()
avg_step_delta = landscape["avg_step_delta"].reshape(GRID_SIZE, GRID_SIZE).cpu().numpy()
path_length = landscape["path_length"].reshape(GRID_SIZE, GRID_SIZE).cpu().numpy()

print("step_deltas:", tuple(landscape["step_deltas"].shape))


In [ ]:
def plot_contour(height, title, cmap="viridis"):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    contour = ax.contourf(xx.cpu().numpy(), yy.cpu().numpy(), height, levels=24, cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("Direction u")
    ax.set_ylabel("Direction v")
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label("height")
    fig.tight_layout()
    plt.show()


plot_contour(final_step_delta, "Landscape height = ||f_T - f_{T-1}||_2 at final step", cmap="magma")
plot_contour(avg_step_delta, "Landscape height = average ||f_t - f_{t-1}||_2", cmap="plasma")


In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(xx.cpu().numpy(), yy.cpu().numpy(), final_step_delta, cmap="magma", linewidth=0, antialiased=True)
ax.set_title("3D surface: final-step residual landscape")
ax.set_xlabel("Direction u")
ax.set_ylabel("Direction v")
ax.set_zlabel("||f_T - f_{T-1}||_2")
fig.tight_layout()
plt.show()


In [ ]:
center_idx = num_points // 2
center_curve = landscape["step_deltas"][center_idx].cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, NUM_STEPS + 1), center_curve, marker="o")
ax.set_title("Center point convergence curve")
ax.set_xlabel("Step t")
ax.set_ylabel("||f_t - f_{t-1}||_2")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()
